# Smart Resume & Job Matching RAG System

**Modern Data Engineering for Advanced AI Systems — SDAIA Academy**

A Retrieval-Augmented Generation (RAG) application that reads a PDF resume, retrieves the most relevant resume sections for a job description, and generates an AI-based match recommendation.


## 1. Install dependencies


In [ ]:
!pip install -q openai docling

In [ ]:
!pip install -q gradio

## 2. Configure OpenRouter
The API key is loaded securely from Google Colab Secrets using the name `OPENROUTER_API_KEY`. The key itself is not stored in this notebook.


In [ ]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

print("OpenRouter client ready!")

## 3. Upload and parse the resume
Upload a PDF resume. Docling extracts the document content and converts it to Markdown text.


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from docling.document_converter import DocumentConverter

cv_filename = list(uploaded.keys())[0]

converter = DocumentConverter()
result = converter.convert(cv_filename)

cv_text = result.document.export_to_markdown()

print(cv_text)

## 4. Chunk the resume
The extracted resume text is split into smaller chunks so retrieval can find the sections most relevant to a job description.


In [ ]:
def split_text(text, chunk_size=500):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

chunks = split_text(cv_text, chunk_size=100)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

## 5. Create embeddings
Each resume chunk is converted into a vector embedding for semantic similarity search.


In [ ]:
chunk_embeddings = []

for chunk in chunks:
    response = client.embeddings.create(
        model="liquid/lfm-2.5-embedding-350m:free",
        input=chunk,
        encoding_format="float"
    )

    chunk_embeddings.append(response.data[0].embedding)

print("Embeddings created for all chunks!")
print("Number of embeddings:", len(chunk_embeddings))
print("Vector length:", len(chunk_embeddings[0]))

## 6. Semantic similarity


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


## 7. RAG job-matching function
The job description is embedded, compared with all resume chunks, and the top 3 chunks are retrieved. Those chunks augment the prompt sent to the LLM, which generates the final recommendation.


In [ ]:
def analyze_job(job_description):

    # 1. Convert job description to embedding
    job_response = client.embeddings.create(
        model="liquid/lfm-2.5-embedding-350m:free",
        input=job_description,
        encoding_format="float"
    )

    job_embedding = job_response.data[0].embedding

    # 2. Compare job description with all resume chunks
    scores = []

    for emb in chunk_embeddings:
        score = cosine_similarity(job_embedding, emb)
        scores.append(score)

    # 3. Retrieve the top 3 most relevant chunks
    top_indices = np.argsort(scores)[-3:][::-1]

    retrieved_chunks = [chunks[i] for i in top_indices]

    retrieved_context = "\n\n".join(retrieved_chunks)

    # 4. Ask the AI to evaluate the match
    prompt = f"""
You are an AI recruitment assistant.

Evaluate the candidate using ONLY the retrieved resume information.

JOB DESCRIPTION:
{job_description}

RETRIEVED RESUME INFORMATION:
{retrieved_context}

Provide:
1. Recommendation: Strong Match, Moderate Match, or Weak Match
2. Matching Skills
3. Missing Skills
4. Short Explanation

Do not invent information.
"""

    response = client.chat.completions.create(
        model="inclusionai/ling-3.0-flash-fin:free",
        messages=[
            {"role": "user", "content": prompt}
        ],
        extra_body={"reasoning": {"enabled": False}}
    )

    return response.choices[0].message.content

## 8. Launch the Gradio application
Enter any job description in the interface and select **Submit** to receive the AI recommendation.


In [ ]:
import gradio as gr

app = gr.Interface(
    fn=analyze_job,
    inputs=gr.Textbox(
        lines=10,
        label="Job Description",
        placeholder="Enter the job description here..."
    ),
    outputs=gr.Markdown(label="AI Recommendation"),
    title="Smart Resume & Job Matching RAG System",
    description="Enter a job description to evaluate its match with the uploaded resume."
)

app.launch()